# IALS (Implicit Alternating Least Squares )

В этом ноутбуке я построю IALS модель, протестирую её и сохраню 

## Импорт библиотек 

In [25]:
import pandas as pd
import numpy as np
import scipy 
import os
import implicit 
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from implicit.evaluation import train_test_split as implicit_split 
import implicit.gpu

In [16]:
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## Загрузка данных

#### Выгружаем матрицу

In [7]:
results_path = Path("../../../results/matrices")
user_item_sparse = scipy.sparse.load_npz(results_path/'artnail_user_item_sparse.npz')

#### Выгружаем очищенные таблицы

In [8]:
clean_path = Path("../../../Tables/CleanTable")
users_clean = pd.read_csv(clean_path / 'users_clean_final.csv')
items_clean = pd.read_csv(clean_path / 'items_clean_final.csv')

In [14]:
f'Размеры матрицы: {user_item_sparse.shape}'

'Размеры матрицы: (2893, 258)'

## IALS model 

#### Train/Validation split для метрик

In [35]:
train_sparse, val_sparse = implicit_split(user_item_sparse, train_percentage=0.9, random_state=42)

In [36]:
print(f"  Train: {train_sparse.nnz} ({train_sparse.nnz/user_item_sparse.nnz:.1%})")
print(f"  Val:   {val_sparse.nnz} ({val_sparse.nnz/user_item_sparse.nnz:.1%})")

  Train: 877 (90.4%)
  Val:   93 (9.6%)


#### Построение базовой модели 

In [37]:
IALS_model = model = implicit.als.AlternatingLeastSquares(
    factors=64,           # Размер эмбеддингов (пользователи × услуги)
    regularization=0.1,   # L2 регуляризация (0.01-0.5)
    iterations=50,        # ALS итераций (сходимость)
    random_state=42,      # Репродуцируемость
    use_gpu=False,        # CPU 
    num_threads=4         # Потоки (4-8 оптимально)
)

c:\Users\msgt0\AppData\Local\Programs\Python\Python311\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Обучение 

In [39]:
CONFIDENCE_SCALE = 15
IALS_model.fit(train_sparse*CONFIDENCE_SCALE)

print(f"  User embeddings: {model.user_factors.shape}")
print(f"  Item embeddings: {model.item_factors.shape}")

  0%|          | 0/50 [00:00<?, ?it/s]

  User embeddings: (2893, 64)
  Item embeddings: (258, 64)
